%md
# Dashboard 01 · Power BI tables
Builds small, flat tables for the compliance dashboard: rule recall by pattern, precision by rule, and the alert queue with agent decisions. Stored in aml_eval because they combine results with labels.

In [0]:
from pyspark.sql import functions as F

GOLD = "workspace.aml_gold"
SILVER = "workspace.aml_silver"
EVAL = "workspace.aml_eval"

alerts = spark.table(f"{GOLD}.alerts")
laundering = spark.table(f"{SILVER}.transactions").filter("is_laundering = 1")
patterns = spark.table(f"{EVAL}.pattern_labels").select("txn_id", "pattern_type")

involved = (
    laundering.select(F.col("from_account_id").alias("account_id"))
    .union(laundering.select(F.col("to_account_id").alias("account_id")))
    .distinct()
    .withColumn("is_involved", F.lit(True))
)

alert_truth = (
    alerts
    .join(involved, on="account_id", how="left")
    .fillna(False, subset=["is_involved"])
)

In [0]:
alerted_ids = alerts.select("account_id")

caught = (
    laundering.select("txn_id", "from_account_id", "to_account_id")
    .join(patterns, on="txn_id", how="left")
    .fillna("UNCLASSIFIED", subset=["pattern_type"])
    .join(alerted_ids.withColumnRenamed("account_id", "from_account_id").withColumn("from_alerted", F.lit(True)), on="from_account_id", how="left")
    .join(alerted_ids.withColumnRenamed("account_id", "to_account_id").withColumn("to_alerted", F.lit(True)), on="to_account_id", how="left")
    .fillna(False, subset=["from_alerted", "to_alerted"])
    .withColumn("caught", F.col("from_alerted") | F.col("to_alerted"))
)

recall_by_pattern = (
    caught.groupBy("pattern_type")
    .agg(
        F.count("*").alias("laundering_txns"),
        F.avg(F.col("caught").cast("int")).alias("rule_recall"),
    )
)

(
    recall_by_pattern.write
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{EVAL}.rule_recall_by_pattern")
)

display(recall_by_pattern.orderBy(F.desc("rule_recall")))

In [0]:
precision_by_rule = (
    alert_truth
    .withColumn("rule", F.explode("triggered_rules"))
    .groupBy("rule")
    .agg(
        F.count("*").alias("alerts_with_rule"),
        F.avg(F.col("is_involved").cast("int")).alias("precision"),
    )
)

(
    precision_by_rule.write
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{EVAL}.rule_precision_by_rule")
)

display(precision_by_rule.orderBy(F.desc("precision")))

In [0]:
agent = spark.table(f"{EVAL}.agent_results").select(
    "alert_id", "agent_decision", "agent_pattern", "agent_confidence", "risk_score"
)

queue = (
    alert_truth
    .join(agent, on="alert_id", how="left")
    .select(
        "alert_id",
        "account_id",
        "priority",
        "rules_fired",
        F.array_join("triggered_rules", ", ").alias("triggered_rules"),
        F.round("in_amount_usd", 2).alias("in_amount_usd"),
        F.round("out_amount_usd", 2).alias("out_amount_usd"),
        F.round("pass_through_ratio", 3).alias("pass_through_ratio"),
        "in_degree",
        "out_degree",
        "is_involved",
        F.col("agent_decision").isNotNull().alias("agent_reviewed"),
        "agent_decision",
        "agent_pattern",
        "agent_confidence",
        "risk_score",
    )
)

rows = queue.count()
reviewed = queue.filter("agent_reviewed").count()

print(f"Alert queue rows: {rows:,}")
print(f"Reviewed by agent: {reviewed}")

assert rows == 5473, "Alert count changed"
assert reviewed == 20, "Expected 20 agent-reviewed alerts"

print("Quality gate passed")

(
    queue.write
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{EVAL}.pbi_alert_queue")
)